# Minho Fresh — revenue report 2025 (fixed)
This is Ana's notebook after the fixes of Practical 2, Exercise 1. It runs from top to bottom (**Restart** + **Run All** in VS Code) with the conda environment `ctds`.

Every change is marked in the code with a comment `# FIXED (n)`:

| n | Defect in Ana's notebook | Fix |
|---|---|---|
| 1 | Colab-only code (`drive.mount`, `!pip install` without versions) | removed; packages come from the environment |
| 2 | Cells out of order (analysis before reading) | cells in the order they must run |
| 3 | Absolute Drive paths | relative paths `data/...` |
| 4 | `YEAR` defined in a deleted cell | defined at the top |
| 5 | Missing `import numpy` | imported at the top |
| 6 | Decimal comma: revenue read as text | `decimal=","` |
| 7 | Dates read as text (dd/mm/yyyy) | `pd.to_datetime(..., format="%d/%m/%Y")` |
| 8 | Store S01 twice in the store table | `drop_duplicates` and `validate="many_to_one"` |
| 9 | Sample without seed | `random_state=SEED` |
| 10 | Conclusion not supported by the outputs | rewritten from the outputs |

## Setup: imports, parameters and the data folder

In [ ]:
import sys
from pathlib import Path

import numpy as np          # FIXED (5): numpy was used (np.round) but never imported
import pandas as pd

# FIXED (1): no drive.mount and no '!pip install': the packages come from the conda environment ctds
# FIXED (3): relative paths. data/ is next to the notebook, or one level up when the notebook is in
#            notebooks/ (the repository layout of Exercise 4, used by GitHub Actions)
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    DATA = Path("/content/data")      # only if you ever open it in Colab: upload data/ first
else:
    DATA = Path("data") if Path("data").is_dir() else Path("../data")

YEAR = 2025   # FIXED (4): YEAR was defined in a cell that Ana deleted (execution count [3] was missing)
SEED = 28504  # used in FIXED (9)
print("pandas", pd.__version__, "| data folder:", DATA)

## 0. Read the data with the right types
FIXED (2): this cell comes first. In Ana's notebook the analysis was above it (execution counts 5, 2, 4…).

In [ ]:
# FIXED (6): the file was saved by Excel in Portuguese: ';' between columns AND ',' as the decimal mark.
#            Without decimal="," the revenue is read as text and .sum() glues the strings together.
sales = pd.read_csv(DATA / "store_sales_pt.csv", sep=";", decimal=",")

# FIXED (7): week_start arrives as text like '02/01/2023'. Comparing it with '2025-01-01' compares
#            strings, not dates, and max() returns the largest string (31/07/2023). Convert it to a date.
sales["week_start"] = pd.to_datetime(sales["week_start"], format="%d/%m/%Y")

stores = pd.read_csv(DATA / "stores_dup.csv")
sales.dtypes   # check: revenue_eur float64, week_start datetime64

In [ ]:
# FIXED (8): the store table must have one row per store_id, but S01 (Braga Centro) appears twice,
#            so the join doubled every Braga sale (10 990 rows instead of 9 420).
print("Repeated store_id:", stores.loc[stores["store_id"].duplicated(), "store_id"].tolist())
stores = stores.drop_duplicates(subset="store_id")

# validate="many_to_one" makes pandas stop with an error if the store table ever has repeated keys again
df = sales.merge(stores[["store_id", "city", "size_m2"]], on="store_id",
                 how="left", validate="many_to_one")
assert len(df) == len(sales), "the join changed the number of rows"
print(len(df), "rows")   # control value: 9 420

## 1. Revenue by store in 2025

In [ ]:
# FIXED (7): filter on the year of a real date, not on a text comparison
sales_year = df[df["week_start"].dt.year == YEAR]
total = sales_year["revenue_eur"].sum()
by_store = sales_year.groupby("store_name")["revenue_eur"].sum().sort_values(ascending=False)
print(f"Rows in {YEAR}: {len(sales_year)}")          # 52 weeks x 6 stores x 10 products = 3 120
print(f"Total revenue {YEAR}: {total:,.2f} EUR")
by_store.round(2)

In [ ]:
by_store.plot(kind="bar", title=f"Revenue by store, {YEAR} (EUR)");

## 2. Revenue per m²

In [ ]:
per_m2 = sales_year.groupby("store_name").agg(revenue=("revenue_eur", "sum"),
                                              size_m2=("size_m2", "first"))
per_m2["eur_per_m2"] = np.round(per_m2["revenue"] / per_m2["size_m2"], 2)   # needs FIXED (5)
per_m2.sort_values("eur_per_m2", ascending=False)

## 3. The last week in the data

In [ ]:
# with real dates (FIXED 7), max() is the latest date, not the largest string
last_week = df["week_start"].max()
last_week_revenue = df.loc[df["week_start"] == last_week, "revenue_eur"].sum()
print("Last week:", last_week.date())
print(f"Revenue in the last week: {last_week_revenue:,.2f} EUR")

## 4. Quick check on a sample

In [ ]:
# FIXED (9): Ana's sample(500) had no seed, so the mean changed in every run.
#            With a fixed seed it is the same 500 rows, and the same mean, every time.
sample = sales_year.sample(500, random_state=SEED)
sample_mean = sample["revenue_eur"].mean()
round(sample_mean, 2)

## 5. Control values
If one of these checks fails, the notebook stops with an error (and the GitHub Actions job of Exercise 4 turns red).

In [ ]:
checks = {
    "rows after the join": (len(df), 9420),
    "total revenue 2025": (round(total, 2), 1328819.85),
    "revenue Braga Centro 2025": (round(by_store["Braga Centro"], 2), 289215.49),
    "best store per m2": (per_m2["eur_per_m2"].idxmax(), "Barcelos"),
    "last week": (str(last_week.date()), "2025-12-29"),
    "revenue last week": (round(last_week_revenue, 2), 23655.99),
    "sample mean (seed 28504)": (round(sample_mean, 2), 441.63),
}
for name, (got, expected) in checks.items():
    print(f"{name:28s} {got!s:>14}   {'OK' if got == expected else 'EXPECTED ' + str(expected)}")
assert all(got == expected for got, expected in checks.values()), "control values do not match"

## Conclusion
FIXED (10): Ana wrote her conclusion from wrong outputs (Braga first, 557 088 EUR). From the outputs above:

In 2025 the store with the highest revenue was **Porto Boavista** (310 834 EUR), followed by Braga Centro (289 215 EUR); total revenue of the chain was **1 328 820 EUR**. Per m², the best store was **Barcelos** (169 EUR/m²) and Porto Boavista the lowest (141 EUR/m²).

Ana's numbers came from the repeated Braga row in the store table (it doubled every Braga sale in the join) and from comparing dates stored as text.